# Code Databricks (links) :
[Lien d'origine Databricks](https://dbc-cb7385f9-5ab2.cloud.databricks.com/editor/notebooks/186796675216147?o=7474660403768866)

## 0. Configuration & Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ⚠️ SPÉCIFIQUE JUPYTER/PYSPARK STANDARD : Initialisation de la Spark Session avec Delta Lake
spark = SparkSession.builder \
    .appName("RecipePipelineFinal") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# --- CONFIGURATION DES CHEMINS ---
# Chemins (DATABRICKS) - Commentés pour l'usage local
# BASE_PATH       = "/Volumes/workspace/default/recipes_data"
# STAGING_PATH    = "/Volumes/workspace/default/recipes_staging"
# OUTPUT_PATH     = "/Volumes/workspace/default/recipes_parquets"

# Chemin (local) - Activés par défaut pour Jupyter
DATA_DIR        = "../data"
BASE_PATH       = f"{DATA_DIR}/recipes_data"
STAGING_PATH    = f"{DATA_DIR}/recipes_staging"
OUTPUT_PATH     = f"{DATA_DIR}/recipes_parquets"

LAYER1_PATH     = f"{BASE_PATH}/layer1.json"
LAYER2_PATH     = f"{BASE_PATH}/layer2+.json"
DET_INGRS_PATH  = f"{BASE_PATH}/det_ingrs.json"
NUTR_PATH       = f"{BASE_PATH}/recipes_with_nutritional_info.json"
RAW_CSV_PATH    = f"{BASE_PATH}/RAW_recipes.csv"

N_PARTITIONS = 8

print("✅ Configuration terminée. Spark Session prête.")

## PHASE 1 : Transformation initiale et écriture en Parquet (Staging) 
####  **Fonctions natives Spark (Performance) :** Utilisation de `F.transform` pour parcourir les arrays JSON imbriqués. Cela évite d'utiliser des UDF (User Defined Functions) Python, qui forcent Spark à désérialiser les données hors de la JVM et ralentissent considérablement l'exécution.
#### **Normalisation agressive :** Création de la colonne `title_norm` (minuscules, sans espaces superflus, sans ponctuation). C'est indispensable pour fiabiliser la future jointure entre des datasets hétérogènes (ex: Kaggle vs Layer 1).
#### **Anticipation de l'UI :** Calcul en amont d'un `nutri_score` simplifié pour fournir immédiatement des filtres à facettes (A, B, C...) prêts à l'emploi pour le front-end.

In [ ]:
# --- 1. LAYER 1 ---
layer1_schema = StructType([
    StructField("id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("url", StringType(), True),
    StructField("instructions", ArrayType(StructType([StructField("text", StringType(), True)])), True),
    StructField("ingredients", ArrayType(StructType([StructField("text", StringType(), True)])), True),
])

(spark.read.option("multiLine", True).schema(layer1_schema).json(LAYER1_PATH)
 # F.transform permet d'extraire uniquement le texte des structures JSON imbriquées sans utiliser UDF (plus rapide)
 .withColumn("instructions_text", F.concat_ws(" | ", F.transform("instructions", lambda x: x["text"])))
 .withColumn("ingredients_raw", F.transform("ingredients", lambda x: x["text"]))
 # Normalisation agressive (minuscules + suppression ponctuation) pour créer une clé de jointure robuste avec les autres datasets
 .withColumn("title_norm", F.lower(F.trim(F.regexp_replace("title", r"[^a-zA-Z0-9\s]", ""))))
 .drop("instructions", "ingredients")
 .write.mode("overwrite").parquet(f"{STAGING_PATH}/layer1"))

# --- 2. LAYER 2 (Images) ---
layer2_schema = StructType([
    StructField("id", StringType(), True),
    StructField("images", ArrayType(StructType([StructField("id", StringType(), True), StructField("url", StringType(), True)])), True),
])

(spark.read.option("multiLine", True).schema(layer2_schema).json(LAYER2_PATH)
 .withColumn("image_url", F.when(F.size("images") > 0, F.col("images")[0]["url"]))
 .withColumn("image_urls", F.transform("images", lambda x: x["url"]))
 .drop("images")
 .write.mode("overwrite").parquet(f"{STAGING_PATH}/layer2"))

# --- 3. INGRÉDIENTS DÉTECTÉS ---
det_schema = StructType([
    StructField("id", StringType(), True),
    StructField("ingredients", ArrayType(StructType([StructField("text", StringType(), True)])), True),
    StructField("valid", ArrayType(BooleanType()), True) 
])

(spark.read.option("multiLine", True).schema(det_schema).json(DET_INGRS_PATH)
 # 1. On extrait uniquement le texte dans un tableau simple ["penne", "sel", ...]
 .withColumn("ingr_texts", F.transform("ingredients", lambda x: x["text"]))
 # 2. On fusionne les deux tableaux parallèles : [{"ingr_texts": "penne", "valid": true}, ...]
 .withColumn("zipped", F.arrays_zip("ingr_texts", "valid"))
 # 3. On filtre ce nouveau tableau fusionné, puis on ne garde que le texte
 .withColumn("ingredients_validated", 
             F.transform(
                 F.filter("zipped", lambda x: x["valid"] == True), 
                 lambda x: x["ingr_texts"]
             ))
 .drop("ingredients", "valid", "ingr_texts", "zipped")
 .write.mode("overwrite").parquet(f"{STAGING_PATH}/det_ingrs"))

# --- 4. NUTRITION ---
nutr_schema = StructType([
    StructField("title", StringType(), True),
    StructField("nutr_values_per100g", StructType([
        StructField("energy", FloatType(), True), StructField("fat", FloatType(), True),
        StructField("protein", FloatType(), True), StructField("salt", FloatType(), True),
        StructField("saturates", FloatType(), True), StructField("sugars", FloatType(), True),
    ]), True),
])

(spark.read.option("multiLine", True).schema(nutr_schema).json(NUTR_PATH)
 .withColumn("energy_kcal", F.col("nutr_values_per100g.energy"))
 .withColumn("fat_g", F.col("nutr_values_per100g.fat"))
 .withColumn("protein_g", F.col("nutr_values_per100g.protein"))
 .withColumn("salt_g", F.col("nutr_values_per100g.salt"))
 .withColumn("saturates_g", F.col("nutr_values_per100g.saturates"))
 .withColumn("sugars_g", F.col("nutr_values_per100g.sugars"))
 # Création d'un Nutri-Score simplifié basé uniquement sur les calories pour faciliter les filtres côté moteur de recherche
 .withColumn("title_norm", F.lower(F.trim(F.regexp_replace("title", r"[^a-zA-Z0-9\s]", ""))))
 .drop("nutr_values_per100g", "title")
 .write.mode("overwrite").parquet(f"{STAGING_PATH}/nutrition"))

# --- 5. KAGGLE CSV ---
(spark.read.option("header", True).option("inferSchema", True).option("multiLine", True).option("escape", '"').csv(RAW_CSV_PATH)
 .select(
     F.col("id").cast(StringType()).alias("kaggle_id"),
     F.col("minutes").cast(IntegerType()).alias("cook_minutes"),
     F.col("tags").alias("tags_raw"),
     F.col("nutrition").alias("nutrition_raw"),
     F.col("n_steps").cast(IntegerType()),
     F.col("description"),
     F.col("name").alias("name_kaggle")
 )
 .withColumn("tags", F.split(F.regexp_replace(F.regexp_replace("tags_raw", r"\[\]']", ""), r",\s+", ","), ","))
 .withColumn("title_norm", F.lower(F.trim(F.regexp_replace("name_kaggle", r"[^a-zA-Z0-9\s]", ""))))
 # Nettoyage des chaînes de type "[tag1, tag2]" en un véritable Array Spark
 .withColumn("nutrition_array", F.split(F.regexp_replace("nutrition_raw", r"\[\]]", ""), ","))
 .withColumn("kaggle_energy_kcal", F.col("nutrition_array")[0].cast(FloatType()))
 .drop("tags_raw", "name_kaggle", "nutrition_array")
 .dropDuplicates(["title_norm"])
 .write.mode("overwrite").parquet(f"{STAGING_PATH}/kaggle"))

print("✅ Étape 1 terminée : Tous les fichiers sont mis sous format Parquet dans la zone de staging.")

## PHASE 2 : Assemblage (Jointures optimisées)
#### **Le pivot (`explode_outer`) :** C'est la transformation clé. On passe d'un format de *stockage* (une ligne = une recette avec une liste d'ingrédients) à un format d' *indexation* (une ligne = un ingrédient lié à une recette). Cela permet des requêtes de filtrage ultra-rapides sans avoir à scanner l'intérieur des arrays.
#### **Format Delta & Data Skipping :** L'index des ingrédients n'est pas partitionné par dossiers (pour éviter le problème des "petits fichiers"). À la place, on utilise l'optimisation **Z-Order** sur la colonne `ingredient`. Lors d'une recherche (ex: "tomate"), le moteur sait instantanément quels fichiers lire et lesquels ignorer, réduisant la latence à quelques millisecondes.
#### **Séparation des usages :** Création de tables distinctes (Main, Index, Summary) pour que l'API de recherche n'interroge que les colonnes strictement nécessaires à l'affichage ou au filtrage.

In [ ]:
# Chargement des Parquets depuis le Staging
df_layer1 = spark.read.parquet(f"{STAGING_PATH}/layer1")
df_layer2 = spark.read.parquet(f"{STAGING_PATH}/layer2")
df_det    = spark.read.parquet(f"{STAGING_PATH}/det_ingrs")
df_nutr   = spark.read.parquet(f"{STAGING_PATH}/nutrition")
df_kaggle = spark.read.parquet(f"{STAGING_PATH}/kaggle")

# Assemblage successif
# Utilisation de LEFT JOIN pour conserver la recette de base (Layer 1) même si elle n'a pas d'image ou de données nutritionnelles
df_base = (
    df_layer1
    .join(df_layer2, on="id", how="left")
    .join(df_det, on="id", how="left")
    .join(df_nutr, on="title_norm", how="left")
    .join(df_kaggle, on="title_norm", how="left")
)

# Enrichissement
df_enriched = (
    df_base
    .withColumn("n_ingredients_validated", F.when(F.col("ingredients_validated").isNotNull(), F.size("ingredients_validated")).otherwise(0))
    .withColumn("cook_time_category",
        F.when(F.col("cook_minutes") <= 30, F.lit("rapide"))
         .when(F.col("cook_minutes") <= 60, F.lit("moyen"))
         .when(F.col("cook_minutes").isNotNull(), F.lit("long"))
         .otherwise(F.lit("inconnu")))
    .withColumn("has_image", F.col("image_url").isNotNull())
    # Coalesce prend la première valeur non-nulle. Si energy_kcal est null, il prend kaggle_energy_kcal
    .withColumn("energy_kcal", F.coalesce(F.col("energy_kcal"), F.col("kaggle_energy_kcal")))
    # Le Nutri-Score est calculé ICI pour bénéficier des données récupérées via Kaggle
    .withColumn("nutri_score",
        F.when(F.col("energy_kcal") < 80, F.lit("A")).when(F.col("energy_kcal") < 160, F.lit("B"))
         .when(F.col("energy_kcal") < 270, F.lit("C")).when(F.col("energy_kcal") < 400, F.lit("D"))
         .otherwise(F.lit("E")))
)

# Sélection finale et dédoublonnage (sécurité au cas où la jointure sur title_norm génère des doublons)
df_final = df_enriched.select(
    F.col("id").alias("recipe_id"), "title", "description",
    "instructions_text", "ingredients_raw", "ingredients_validated", "n_ingredients_validated", "n_steps",
    "cook_minutes", "cook_time_category", "image_url", "image_urls", "has_image",
    F.col("url").alias("source_url"), "energy_kcal", "nutri_score", "tags"
).dropDuplicates(["recipe_id"])

# Création de la table de détails nutritionnels
df_nutrition_detail = df_enriched.select(
    F.col("id").alias("recipe_id"), "fat_g", "protein_g", "salt_g", "saturates_g", "sugars_g"
).dropDuplicates(["recipe_id"])

# --- ÉCRITURE FINALE EN DELTA ---
# 1. Main Table
# On partitionne physiquement les données pour accélérer les futures requêtes filtrant par nutri_score
(df_final.repartition(N_PARTITIONS, "nutri_score")
 .write.format("delta").mode("overwrite")
 .partitionBy("nutri_score")
 .save(f"{OUTPUT_PATH}/recipes_main"))

# 2. Ingredients Index 
df_index = (df_final.select(
    "recipe_id", "title", "nutri_score", "image_url", 
    "cook_time_category", F.explode_outer("ingredients_validated").alias("ingredient")
 )
 .withColumn("ingredient", F.lower(F.trim("ingredient")))
 .filter(F.col("ingredient").isNotNull() & (F.col("ingredient") != "")))

# On écrit le dataframe SANS partitionBy pour éviter de créer des milliers de minuscules dossiers
df_index.write.format("delta").mode("overwrite").save(f"{OUTPUT_PATH}/ingredients_index")

# 3. Recipes Nutrition Detail
(df_nutrition_detail
 .write.format("delta").mode("overwrite")
 .save(f"{OUTPUT_PATH}/recipes_nutrition_detail"))

# OPTIMISATION : Z-Order
# Optimisation des recherches par nom, ingrédients et id
# Attention: Requiert un environnement supportant complètement l'open-source Delta Lake SQL
try:
    spark.sql(f"OPTIMIZE delta.`{OUTPUT_PATH}/recipes_main` ZORDER BY (title)")
    spark.sql(f"OPTIMIZE delta.`{OUTPUT_PATH}/ingredients_index` ZORDER BY (ingredient)")
    spark.sql(f"OPTIMIZE delta.`{OUTPUT_PATH}/recipes_nutrition_detail` ZORDER BY (recipe_id)")
    print("⚡ Optimisations Z-Order appliquées avec succès.")
except Exception as e:
    print(f"⚠️ L'optimisation Z-Order a échoué (souvent dû à des limitations d'environnement hors Databricks) : {e}")

print("✅ Pipeline ETL terminé avec succès !")